# Agentic Pipeline for English-to-Chinese Dialogue Summarization

This notebook implements a local agentic pipeline for English-to-Chinese cross-lingual dialogue summarization.

The pipeline uses three local small language model agents. Agent 1 analyzes the original English dialogue and extracts a structured semantic representation with resolved intents, speech acts, semantic roles, and evidence. Agent 2 selects the key events from this semantic representation and generates a concise Chinese summary. Agent 3 verifies and revises the Chinese summary for conciseness, naturalness, and factuality.

```text
English Dialogue
→ Agent 1: Semantic Understanding and Disambiguation Agent
→ Agent 2: Summary Generation Agent
→ Agent 3: Verification and Revision Agent
→ Final Chinese Summary
```

The pipeline consists of three agents:

```text
Agent 1: Semantic Understanding and Disambiguation Agent
Input: original English dialogue
Output: structured semantic representation with resolved intents, speech acts, semantic roles, and evidence

Agent 2: Summary Generation Agent
Input: structured semantic representation from Agent 1
Output: key events selection and Chinese summary

Agent 3: Verification and Revision Agent
Input: Chinese summary from Agent 2
Output: revised final Chinese summary
```
The local small language models are served through Ollama. The notebook controls the agent workflow, prompt design, input/output processing, JSON parsing, intermediate output inspection, and result saving.

## 0. Local Ollama Setup

Before running this notebook, install Ollama and download the model locally.

### Recommended model setup

This notebook uses a single-model multi-agent setup. All three agents use Qwen3.5-27B.

```bash
ollama pull qwen3.5:27b
```

You can check downloaded models with:

```bash
ollama list
```

You can check currently loaded models with:

```bash
ollama ps
```

If the Ollama server is not running, start it with:

```bash
ollama serve
```

On macOS, opening the Ollama app usually starts the local server automatically.


In [1]:
# Cell 1: Install required Python packages.
# Run this only once if the packages are not installed

!pip install requests pandas tqdm



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# current working path check
import os
from pathlib import Path

PROJECT_ROOT = Path("/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization")
os.chdir(PROJECT_ROOT)

print("Current working directory:", Path.cwd())

Current working directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization


In [3]:
# Cell 2: Imports and global configuration

import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_HOST = "http://localhost:11434"

# Agent models
# Efficient Qwen-family multi-size setup:
# Agent 1/2 use qwen3.5:9b for faster semantic extraction and draft summary generation.
# Agent 3 uses qwen3.5:27b as a stronger verifier and final reviser.
SEMANTIC_UNDERSTANDING_MODEL = "qwen3.5:9b"
SUMMARY_GENERATION_MODEL = "qwen3.5:9b"
REVISION_MODEL = "qwen3.5:27b"

DEFAULT_TEMPERATURE = 0.2

# Smaller context is faster and usually enough for XSAMSum-style dialogues.
DEFAULT_NUM_CTX = 4096
REVISION_NUM_CTX = 4096

# Input dataset path
RAW_TEST_PATH = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/raw/test.json"
)

# Output directory
OUTPUT_DIR = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output files
INTERMEDIATE_OUTPUT_PATH = OUTPUT_DIR / "3agents_0shot_qwen9b_agent12_intermediate.jsonl"
FULL_OUTPUT_PATH = OUTPUT_DIR / "3agents_0shot_qwen9b_agent12_qwen27b_agent3_final.jsonl"
FINAL_CSV_PATH = OUTPUT_DIR / "3agents_0shot_qwen9b_agent12_qwen27b_agent3_final.csv"
ERROR_OUTPUT_PATH = OUTPUT_DIR / "3agents_0shot_qwen9b_agent12_qwen27b_agent3_errors.jsonl"

print("Raw test path:", RAW_TEST_PATH)
print("Output directory:", OUTPUT_DIR)
print("Intermediate JSONL output path:", INTERMEDIATE_OUTPUT_PATH)
print("Full JSONL output path:", FULL_OUTPUT_PATH)
print("Final CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

print("Agent 1 model:", SEMANTIC_UNDERSTANDING_MODEL)
print("Agent 2 model:", SUMMARY_GENERATION_MODEL)
print("Agent 3 model:", REVISION_MODEL)

Raw test path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/raw/test.json
Output directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results
Intermediate JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/3agents_0shot_qwen9b_agent12_intermediate.jsonl
Full JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/3agents_0shot_qwen9b_agent12_qwen27b_agent3_final.jsonl
Final CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/3agents_0shot_qwen9b_agent12_qwen27b_agent3_final.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/3agents_0shot_qwen9b_agent12_qwen27b_agent3_errors.jsonl
Agent 1 model: qwen3.5:9b
Agent 2 model: qwen3.5:9b
Agent 3 model: qwen3.5:27b


In [4]:
# Cell 3: Check whether Ollama is running

def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        response.raise_for_status()
        models = response.json().get("models", [])

        print("Ollama server is running.")
        print(f"Downloaded models: {[m.get('name') for m in models]}")

        return True

    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))

        return False


_ = check_ollama_server()

Ollama server is running.
Downloaded models: ['qwen3.5:9b', 'qwen3.5:27b']


In [5]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 900,
    keep_alive: Optional[str] = None,
    json_mode: bool = True,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    if json_mode:
        payload["format"] = "json"

    if keep_alive is not None:
        payload["keep_alive"] = keep_alive

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()


def load_ollama_model(model: str) -> None:
    """Preload a model into Ollama memory."""
    payload = {
        "model": model,
        "messages": [],
        "stream": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=300,
    )
    response.raise_for_status()

    print(f"Loaded model: {model}")


def unload_ollama_model(model: str) -> None:
    """Unload a model from Ollama memory."""
    payload = {
        "model": model,
        "messages": [],
        "keep_alive": 0,
        "stream": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=300,
    )
    response.raise_for_status()

    print(f"Unloaded model: {model}")

## 1. Prompt Templates

Each agent is defined as:

```text
Agent = model + role-specific prompt + input/output format
```

In the first version, we use a fixed workflow rather than a fully autonomous agent system.


In [6]:
# Cell 5: Prompt templates

# Agent 1: Linguistic Robustness & Disambiguation
SEMANTIC_UNDERSTANDING_PROMPT = """Analyze this English dialogue with a focus on pragmatic competence and disambiguation.

Your goal is to extract only key events that affect the final outcome.
Do not analyze every utterance.

Extract:
- Participants: List speakers.
- Illocutionary acts: Identify the actual intent behind speech, such as request, suggestion, command, indirect refusal, acceptance, reminder, cancellation, or changed plan.
- Semantic roles: Define who did what to whom clearly.
- Contextual evidence: Use short quotes that justify your interpretation.

Focus:
Resolve ambiguity in context. The true intent may be resolved across multiple turns.

Prioritize:
- final decisions
- changed plans
- cancellations
- refusals
- commitments
- reminders
- non-literal or indirect meanings
- final outcome of the dialogue

Do not include:
- greetings
- jokes
- repeated reactions
- conversation closings
unless they affect the final outcome.

Important actor/action rules:
- Pay special attention to imperatives and directives.
- If speaker A says an imperative such as "Just text him" to speaker B, the actor of the action is speaker B, not speaker A.
- Do not infer that the speaker will perform the action unless they explicitly say "I will" or clearly accept responsibility.
- If a later utterance overrides an earlier preference, use the later utterance to determine the final outcome.

Placeholder rules:
- Do not infer visual or emotional details from placeholders such as <file_photo>, <file_gif>, or <file>.
- If visual content is unavailable, refer to it only as "the item", "the option", or "the attachment".
- Do not use placeholders as evidence unless the surrounding text explains their meaning.

Extract only 3-6 key semantic events.

Output valid JSON only.
Do not wrap the JSON in markdown.
Do not copy placeholder values such as "string".

Output schema:
{
  "participants": ["string"],
  "semantic_grounding": [
    {
      "event_id": 1,
      "speaker": "string",
      "speech_act": "string",
      "intended_meaning": "string",
      "actor": "string or null",
      "action": "string",
      "object": "string or null",
      "recipient": "string or null",
      "evidence": ["string"]
    }
  ],
  "final_outcome": "string"
}

Input dialogue:
{dialogue}

JSON output:
"""


# Agent 2: Selection and generation
SUMMARY_GENERATION_PROMPT = """You will receive a structured representation of an English dialogue. Produce a concise Chinese summary.

Process:

1. SELECTION
From the semantic_grounding list, select salient events necessary for coherent understanding.

Prioritize:
- speech acts that drive the dialogue forward, such as commitments, decisions, refusals, cancellations, and changed plans
- events whose intended_meaning reveals non-literal content
- the final outcome of the dialogue
- the final criterion for a decision, if the dialogue is about choosing among options

If an earlier plan is later changed, cancelled, or replaced, summarize the final updated state, not the initial plan.

Target only 1-3 events.

2. CHINESE GENERATION
Write a coherent Chinese summary based on the selected events.

Requirements:
- Faithfully represent intended_meaning.
- Preserve participant relationships from actor/recipient roles.
- Emphasize the final outcome.
- Do not mechanically translate English structure.
- Use natural Chinese.
- Keep personal names consistent.
- Do not invent visual details from placeholders such as <file_photo>, <file_gif>, or <file>.
- For simple dialogues, prefer 20-50 Chinese characters.
- For complex dialogues with changed plans, cancellations, or multiple critical events, allow up to 80 Chinese characters.

Output valid JSON only.
Do not wrap the JSON in markdown.
Do not copy placeholder values such as "string".

Output schema:
{
  "selected_events": [
    {
      "event_id": 1,
      "key_event": "string"
    }
  ],
  "summary_zh": "string"
}

Input structured semantic representation:
{semantic_representation}

JSON output:
"""


# Agent 3: Verification and Revision
REVISION_PROMPT = """Evaluate a draft Chinese summary based on the original English dialogue.

You will receive:
- the original English dialogue
- selected_events from Agent 2
- summary_zh, the draft Chinese summary from Agent 2

Generate a revised Chinese summary if one of the criteria is not met.
You may drop events if needed, but log which ones.

Evaluate the draft on the following criteria:

1. FAITHFULNESS
- Flag hallucinated information.
- Flag misinterpreted events or intentions.
- Flag incorrect actor/action/object/recipient relationships.
- Flag cases where the summary assigns an action to the wrong person.
- Pay special attention to imperatives and directives.
- If speaker A says an imperative such as "Just text him" to speaker B, the actor of the action is speaker B, not speaker A.
- Do not infer that the speaker performs the action unless they explicitly say so.

2. COVERAGE
- Flag if the summary does not cover critical events, actions, participants, final decisions, cancellations, or changed plans.
- If an earlier plan is later changed, make sure the summary reflects the final updated outcome.
- If the dialogue is about choosing among options, make sure the summary captures the final decision criterion, such as quality, price, time, or availability.

3. CONCISENESS
- Flag unnecessary details.
- Prefer one concise Chinese sentence.
- For simple dialogues, keep the summary short.
- For complex dialogues with multiple changed plans, allow a slightly longer summary if needed for coverage.

4. NATURALNESS
- Flag awkward Chinese phrasing.
- Flag English syntactic transfer.
- Prefer fluent and natural Chinese.

Revision rules:
- If the draft is correct, keep it unchanged.
- If the draft misses the final outcome, revise it.
- If the draft contains wrong actor/action information, revise it.
- If the draft includes an early plan that was later changed or cancelled, revise it.
- If the draft invents visual details from placeholders such as <file_photo>, <file_gif>, or <file>, revise it.
- Do not add unsupported information.
- Do not over-explain minor details.

Consistency rules:
- If you change summary_zh_final in any way, set needs_revision to true.
- If needs_revision is false, summary_zh_final must be exactly the same as the input summary_zh.
- revision_reason must briefly explain the actual reason for revision.
- If no revision is needed, revision_reason must be "".
- Do not copy placeholder values such as "string".
- issues_identified must be an empty list if no issues are found.
- events_dropped must be an empty list if no events are dropped.

Output valid JSON only.
Do not wrap the JSON in markdown.

Output schema:
{
  "needs_revision": false,
  "issues_identified": [],
  "events_dropped": [],
  "revision_reason": "",
  "summary_zh_final": "string"
}

Original English dialogue:
{dialogue}

Draft Chinese summary and selected events:
{summary_points}

JSON output:
"""

In [7]:
# Cell 6: JSONL utility functions

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def load_processed_ids(path: Path) -> set:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)

    return {str(record["id"]) for record in records if "id" in record}

In [8]:
# Cell 7: Agent functions

import re

def fill_prompt(template: str, replacements: Dict[str, str]) -> str:
    """Replace only named placeholders while keeping JSON braces in the prompt unchanged."""
    prompt = template

    for key, value in replacements.items():
        prompt = prompt.replace("{" + key + "}", value)

    return prompt


def extract_json_object(text: str) -> Dict[str, Any]:
    """Extract and parse a JSON object from a model response.

    This function is intentionally robust because small local models may:
    - wrap JSON in markdown fences,
    - add extra text before or after JSON,
    - output Python-style booleans such as True/False.
    """
    raw = text.strip()

    # Remove markdown fences if present.
    if raw.startswith("```"):
        lines = raw.splitlines()
        raw = "\n".join(
            line for line in lines
            if not line.strip().startswith("```")
        ).strip()

    candidates = [raw]

    # Try extracting the substring between the first "{" and the last "}".
    start = raw.find("{")
    end = raw.rfind("}")

    if start != -1 and end != -1 and end > start:
        candidates.append(raw[start:end + 1])

    for candidate in candidates:
        # First try direct JSON parsing.
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            pass

        # Repair common Python-style JSON issues.
        repaired = candidate
        repaired = re.sub(r"\bTrue\b", "true", repaired)
        repaired = re.sub(r"\bFalse\b", "false", repaired)
        repaired = re.sub(r"\bNone\b", "null", repaired)

        try:
            parsed = json.loads(repaired)
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            pass

    # Safe fallback for debugging.
    return {
        "raw_output": text,
        "parse_error": True,
    }


def format_json(data: Any) -> str:
    """Format Python objects as readable JSON text."""
    return json.dumps(
        data,
        ensure_ascii=False,
        indent=2,
    )


def extract_final_chinese_summary(revision_result: Dict[str, Any]) -> str:
    """Extract the final Chinese summary from Agent 3's revision output."""
    summary = revision_result.get("summary_zh_final", "")

    if isinstance(summary, list):
        return " ".join(str(item).strip() for item in summary if str(item).strip())

    if isinstance(summary, str):
        return summary.strip()

    if summary:
        return str(summary).strip()

    # Backward-compatible fallback.
    summary = revision_result.get("summary_zh", "")

    if isinstance(summary, str):
        return summary.strip()

    raw_output = revision_result.get("raw_output", "")

    if isinstance(raw_output, str):
        return raw_output.strip()

    return ""


def semantic_understanding_agent(dialogue: str) -> Dict[str, Any]:
    """Agent 1: Information Extraction Agent.

    Input:
        Original English dialogue

    Output:
        Structured representation of the dialogue
    """
    prompt = fill_prompt(
        SEMANTIC_UNDERSTANDING_PROMPT,
        {
            "dialogue": dialogue,
        },
    )

    response = call_ollama(
        model=SEMANTIC_UNDERSTANDING_MODEL,
        prompt=prompt,
        temperature=0.1,
        num_ctx=DEFAULT_NUM_CTX,
        keep_alive="10m",
        json_mode=True,
    )

    return extract_json_object(response)

def summary_generation_agent(
    semantic_representation: Dict[str, Any],
) -> Dict[str, Any]:
    """Agent 2: Summary Generation Agent.

    Input:
        Structured semantic representation from Agent 1

    Output:
        Selected key events and draft Chinese summary
    """
    prompt = fill_prompt(
        SUMMARY_GENERATION_PROMPT,
        {
            "semantic_representation": format_json(semantic_representation),
        },
    )

    response = call_ollama(
        model=SUMMARY_GENERATION_MODEL,
        prompt=prompt,
        temperature=0.2,
        num_ctx=DEFAULT_NUM_CTX,
        keep_alive="10m",
        json_mode=True,
    )

    return extract_json_object(response)


def revision_agent(
    dialogue: str,
    summary_generation_result: Dict[str, Any],
) -> Dict[str, Any]:
    """Agent 3: Verification and Revision Agent.

    Input:
        Original dialogue, selected events, and draft Chinese summary from Agent 2

    Output:
        Revised final Chinese summary
    """
    prompt = fill_prompt(
        REVISION_PROMPT,
        {
            "dialogue": dialogue,
            "summary_points": format_json(summary_generation_result),
        },
    )

    response = call_ollama(
        model=REVISION_MODEL,
        prompt=prompt,
        temperature=0.1,
        num_ctx=REVISION_NUM_CTX,
        keep_alive="10m",
        json_mode=True,
    )

    return extract_json_object(response)

## 2. Agent Functions

Each function corresponds to one agent in the pipeline.


In [9]:
# Cell 8: Three-agent pipeline

def run_agents_1_2(example: Dict[str, Any]) -> Dict[str, Any]:
    """Run Agent 1 and Agent 2 with qwen3.5:9b and save intermediate outputs."""

    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_english_summary = example.get("reference_english_summary", "")
    reference_chinese_summary = example.get("reference_chinese_summary", "")

    # Agent 1: Semantic Understanding and Disambiguation Agent
    semantic_representation = semantic_understanding_agent(dialogue)

    # Agent 2: Summary Generation Agent
    summary_generation = summary_generation_agent(
        semantic_representation=semantic_representation,
    )

    return {
        "id": sample_id,
        "dialogue": dialogue,
        "reference_english_summary": reference_english_summary,
        "reference_chinese_summary": reference_chinese_summary,

        "agent1_model": SEMANTIC_UNDERSTANDING_MODEL,
        "agent2_model": SUMMARY_GENERATION_MODEL,

        "agent1_semantic_representation": semantic_representation,
        "agent1_semantic_representation_json": format_json(semantic_representation),

        "agent2_summary_generation": summary_generation,
        "agent2_summary_generation_json": format_json(summary_generation),
    }


def run_agent_3_from_intermediate(
    intermediate_record: Dict[str, Any],
) -> Dict[str, Any]:
    """Run Agent 3 with qwen3.5:27b using saved Agent 1/2 intermediate outputs."""

    dialogue = intermediate_record["dialogue"]
    summary_generation = intermediate_record["agent2_summary_generation"]

    # Agent 3: Verification and Revision Agent
    revision = revision_agent(
        dialogue=dialogue,
        summary_generation_result=summary_generation,
    )

    final_chinese_summary = extract_final_chinese_summary(
        revision_result=revision,
    )

    return {
        **intermediate_record,

        "agent3_model": REVISION_MODEL,

        "agent3_revision": revision,
        "agent3_revision_json": format_json(revision),

        "agent3_final_chinese_summary": final_chinese_summary,
        "final_summary": final_chinese_summary,
    }


def run_three_agent_pipeline(example: Dict[str, Any]) -> Dict[str, Any]:
    """Run the full Qwen-family multi-size pipeline for one example."""

    # Stage 1: Agent 1/2 with qwen3.5:9b
    load_ollama_model(SEMANTIC_UNDERSTANDING_MODEL)
    intermediate_record = run_agents_1_2(example)

    # Stage 2: unload qwen3.5:9b before loading qwen3.5:27b
    models_to_unload = {
        SEMANTIC_UNDERSTANDING_MODEL,
        SUMMARY_GENERATION_MODEL,
    }

    for model in models_to_unload:
        if model != REVISION_MODEL:
            unload_ollama_model(model)

    load_ollama_model(REVISION_MODEL)
    final_record = run_agent_3_from_intermediate(intermediate_record)

    return final_record

## 3. Test with Examples

Start with five examples before running the full dataset.  
This is the best way to inspect the intermediate outputs between agents.


In [10]:
# Cell 9: Load top examples from the original test.json file

def load_top_examples_from_test_json(path: Path, n: int = 5) -> List[Dict[str, Any]]:
    """Load the top n examples from the original JSON test file."""
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

    examples = []

    for i, item in enumerate(raw_data[:n]):
        examples.append({
            "id": f"test_{i+1:05d}",
            "dialogue": item["dialogue"],
            "reference_english_summary": item.get("summary", ""),
            "reference_chinese_summary": item.get("summary_zh", ""),
        })

    return examples


test_data = load_top_examples_from_test_json(RAW_TEST_PATH, n=5)

print(f"Loaded {len(test_data)} examples.")
print("First example:")
print(test_data[0])

Loaded 5 examples.
First example:
{'id': 'test_00001', 'dialogue': "Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye", 'reference_english_summary': "Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.", 'reference_chinese_summary': '汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。'}


In [11]:
# Cell 10: Run the three-agent pipeline for the first example from test.json

result = run_three_agent_pipeline(test_data[0])
result

Loaded model: qwen3.5:9b
Unloaded model: qwen3.5:9b
Loaded model: qwen3.5:27b


{'id': 'test_00001',
 'dialogue': "Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye",
 'reference_english_summary': "Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.",
 'reference_chinese_summary': '汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。',
 'agent1_model': 'qwen3.5:9b',
 'agent2_model': 'qwen3.5:9b',
 'agent1_semantic_representation': {'participants': ['Hannah', 'Amanda'],
  'semantic_grounding': [{'event_id': 1,
    'speaker': 'Hannah',
    'speech_act': 'request',
    'intended_meaning': "Hannah wants to obtain Betty's phone number.",
    'actor': 'Amanda',
    'action': 'search for contact info

In [12]:
# Cell 11: Print three-agent pipeline result clearly

def print_three_agent_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Agent 1: Semantic Understanding and Disambiguation ===")
    print(result["agent1_semantic_representation_json"])
    print()

    print("=== Agent 2: Summary Generation ===")
    print(result["agent2_summary_generation_json"])
    print()

    print("=== Agent 3: Verification and Revision ===")
    print(result["agent3_revision_json"])
    print()

    print("=== Final Chinese Summary ===")
    print(result["final_summary"])
    print()

    print("=== Reference English Summary ===")
    print(result["reference_english_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])


print_three_agent_result(result)

=== Original Dialogue ===
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

=== Agent 1: Semantic Understanding and Disambiguation ===
{
  "participants": [
    "Hannah",
    "Amanda"
  ],
  "semantic_grounding": [
    {
      "event_id": 1,
      "speaker": "Hannah",
      "speech_act": "request",
      "intended_meaning": "Hannah wants to obtain Betty's phone number.",
      "actor": "Amanda",
      "action": "search for contact information",
      "object": "Betty's phone number",
      "recipient": "Hannah",
      "evidence": [
        "Hannah: Hey, do you have Betty's number?",
        "Amanda: Lemme check"
      ]
    },
    {


## 4. Save Results

This saves all intermediate outputs and the final output.


In [13]:
# Cell 12: Reset previous outputs before batch inference

INTERMEDIATE_OUTPUT_PATH.unlink(missing_ok=True)
FULL_OUTPUT_PATH.unlink(missing_ok=True)
FINAL_CSV_PATH.unlink(missing_ok=True)
ERROR_OUTPUT_PATH.unlink(missing_ok=True)

print("Previous output files reset.")
print("Intermediate output path:", INTERMEDIATE_OUTPUT_PATH)
print("Final JSONL output path:", FULL_OUTPUT_PATH)
print("CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Previous output files reset.
Intermediate output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/3agents_0shot_qwen9b_agent12_intermediate.jsonl
Final JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/3agents_0shot_qwen9b_agent12_qwen27b_agent3_final.jsonl
CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/3agents_0shot_qwen9b_agent12_qwen27b_agent3_final.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/3agents_0shot_qwen9b_agent12_qwen27b_agent3_errors.jsonl


## 5. Batch Inference with Checkpointing

This cell processes examples one by one and appends each completed result to `results/agentic_outputs.jsonl`.

If the notebook stops, already processed examples remain saved.


In [ ]:
# Cell 13: Two-stage batch inference with Qwen-family multi-size setup
# Stage 1: Run Agent 1 and Agent 2 with qwen3.5:9b and save intermediate outputs
# Stage 2: Unload qwen3.5:9b, load qwen3.5:27b, and run Agent 3
# Time stamp: 4m 29s

from datetime import datetime

MAX_EXAMPLES = 5
SLEEP_SECONDS = 0.2

subset = test_data[:MAX_EXAMPLES]

print("Batch inference started at:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Qwen-family multi-size setup")
print("Agent 1 model:", SEMANTIC_UNDERSTANDING_MODEL)
print("Agent 2 model:", SUMMARY_GENERATION_MODEL)
print("Agent 3 model:", REVISION_MODEL)

# -------------------------
# Stage 1: Agent 1 + Agent 2 with qwen3.5:9b
# -------------------------

processed_intermediate_ids = load_processed_ids(INTERMEDIATE_OUTPUT_PATH)
print(f"Already processed by Agent 1/2: {len(processed_intermediate_ids)} examples")

load_ollama_model(SEMANTIC_UNDERSTANDING_MODEL)

for ex in tqdm(
    subset,
    desc=f"Stage 1: Agent 1/2 with {SEMANTIC_UNDERSTANDING_MODEL}",
):
    sample_id = str(ex.get("id", "unknown"))

    if sample_id in processed_intermediate_ids:
        continue

    try:
        intermediate_record = run_agents_1_2(ex)
        append_jsonl(intermediate_record, INTERMEDIATE_OUTPUT_PATH)
        processed_intermediate_ids.add(sample_id)
        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "stage": "agent_1_2",
            "id": sample_id,
            "error": repr(e),
            "dialogue": ex.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error in Agent 1/2 on {sample_id}: {repr(e)}")


# -------------------------
# Unload qwen3.5:9b before Agent 3
# -------------------------

models_to_unload = {
    SEMANTIC_UNDERSTANDING_MODEL,
    SUMMARY_GENERATION_MODEL,
}

for model in models_to_unload:
    if model != REVISION_MODEL:
        unload_ollama_model(model)


# -------------------------
# Stage 2: Agent 3 with qwen3.5:27b
# -------------------------

intermediate_records = load_jsonl(INTERMEDIATE_OUTPUT_PATH)

processed_final_ids = load_processed_ids(FULL_OUTPUT_PATH)
print(f"Already processed by Agent 3: {len(processed_final_ids)} examples")

load_ollama_model(REVISION_MODEL)

for intermediate_record in tqdm(
    intermediate_records,
    desc=f"Stage 2: Agent 3 with {REVISION_MODEL}",
):
    sample_id = str(intermediate_record.get("id", "unknown"))

    if sample_id in processed_final_ids:
        continue

    try:
        final_record = run_agent_3_from_intermediate(intermediate_record)
        append_jsonl(final_record, FULL_OUTPUT_PATH)
        processed_final_ids.add(sample_id)
        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "stage": "agent_3",
            "id": sample_id,
            "error": repr(e),
            "dialogue": intermediate_record.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error in Agent 3 on {sample_id}: {repr(e)}")

print("Batch inference finished at:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Finished. Intermediate outputs saved to: {INTERMEDIATE_OUTPUT_PATH}")
print(f"Finished. Final outputs saved to: {FULL_OUTPUT_PATH}")

Batch inference started at: 2026-05-06 19:31:26
Qwen-family multi-size setup
Agent 1 model: qwen3.5:9b
Agent 2 model: qwen3.5:9b
Agent 3 model: qwen3.5:27b
Already processed by Agent 1/2: 0 examples
Loaded model: qwen3.5:9b


Stage 1: Agent 1/2 with qwen3.5:9b:   0%|          | 0/5 [00:00<?, ?it/s]

Unloaded model: qwen3.5:9b
Already processed by Agent 3: 0 examples
Loaded model: qwen3.5:27b


Stage 2: Agent 3 with qwen3.5:27b:   0%|          | 0/5 [00:00<?, ?it/s]

Batch inference finished at: 2026-05-06 19:35:55
Finished. Intermediate outputs saved to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/3agents_0shot_qwen9b_agent12_intermediate.jsonl
Finished. Final outputs saved to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/3agents_0shot_qwen9b_agent12_qwen27b_agent3_final.jsonl


## 6. Export Final Summaries to CSV

This file can be used for ROUGE, BERTScore, OmniScore, or manual analysis.


In [15]:
# Cell 14: Export final summaries to CSV

records = load_jsonl(FULL_OUTPUT_PATH)

rows = []

for record in records:
    if "final_summary" not in record:
        continue

    rows.append({
        "id": record.get("id", ""),
        "dialogue": record.get("dialogue", ""),
        "final_summary": record.get("final_summary", ""),
        "reference_english_summary": record.get("reference_english_summary", ""),
        "reference_chinese_summary": record.get("reference_chinese_summary", ""),

        "agent1_semantic_representation_json": record.get(
            "agent1_semantic_representation_json", ""
        ),
        "agent2_summary_generation_json": record.get(
            "agent2_summary_generation_json", ""
        ),
        "agent3_revision_json": record.get(
            "agent3_revision_json", ""
        ),
        "agent3_final_chinese_summary": record.get(
            "agent3_final_chinese_summary", ""
        ),
    })

df = pd.DataFrame(rows)

if not df.empty:
    df = df.drop_duplicates(subset=["id"], keep="last")

df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"Saved final summaries to: {FINAL_CSV_PATH}")
df

Saved final summaries to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/3agents_0shot_qwen9b_agent12_qwen27b_agent3_final.csv


,id,dialogue,final_summary,reference_english_summary,reference_chinese_summary,agent1_semantic_representation_json,agent2_summary_generation_json,agent3_revision_json,agent3_final_chinese_summary
0,test_00001,"Hannah: Hey, do you have Betty's number?\nAman...",Amanda建议Hannah向Larry索要Betty的号码，尽管Hannah起初不愿，最终...,Hannah needs Betty's number but Amanda doesn't...,汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。,"{\n ""participants"": [\n ""Hannah"",\n ""Am...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Amanda建议Hannah向Larry索要Betty的号码，尽管Hannah起初不愿，最终...
1,test_00002,Eric: MACHINE!\r\nRob: That's so gr8!\r\nEric:...,Rob 确认脱口秀表演在 YouTube 上，Eric 决定立即观看，Rob 也同意一起看。,Eric and Rob are going to watch a stand-up on ...,埃里克和罗伯要在youtube上看一场单口相声。,"{\n ""participants"": [\n ""Eric"",\n ""Rob""...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Rob 确认脱口秀表演在 YouTube 上，Eric 决定立即观看，Rob 也同意一起看。
2,test_00003,"Lenny: Babe, can you help me with something?\r...",Lenny 请 Bob 帮忙挑裤子，虽因已有紫色裤子而犹豫，但在 Bob 建议多色搭配及优先...,Lenny can't decide which trousers to buy. Bob ...,莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。,"{\n ""participants"": [\n ""Lenny"",\n ""Bob...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Lenny 请 Bob 帮忙挑裤子，虽因已有紫色裤子而犹豫，但在 Bob 建议多色搭配及优先...
3,test_00004,"Will: hey babe, what do you want for dinner to...",Emma 因心情不好拒绝 Will 做晚餐，并婉拒其接送提议，决定独自回家。,Emma will be home soon and she will let Will k...,艾玛很快就会回家，而且她会告诉威尔。,"{\n ""participants"": [\n ""Will"",\n ""Emma...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Emma 因心情不好拒绝 Will 做晚餐，并婉拒其接送提议，决定独自回家。
4,test_00005,"Ollie: Hi , are you in Warsaw\r\nJane: yes, ju...",Ollie 因派对拒绝了晚餐邀请，Jane 随后提议将午餐改为周五课后喝茶，双方达成一致。,Jane is in Warsaw. Ollie and Jane has a party....,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...,"{\n ""participants"": [\n ""Ollie"",\n ""Jan...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": true,\n ""issues_identi...",Ollie 因派对拒绝了晚餐邀请，Jane 随后提议将午餐改为周五课后喝茶，双方达成一致。


In [16]:
# Cell 15: Compare generated Chinese summary with the reference Chinese summary

comparison_columns = [
    "id",
    "final_summary",
    "reference_chinese_summary",
]

comparison_df = df[comparison_columns].copy()

comparison_df

,id,final_summary,reference_chinese_summary
0,test_00001,Amanda建议Hannah向Larry索要Betty的号码，尽管Hannah起初不愿，最终...,汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。
1,test_00002,Rob 确认脱口秀表演在 YouTube 上，Eric 决定立即观看，Rob 也同意一起看。,埃里克和罗伯要在youtube上看一场单口相声。
2,test_00003,Lenny 请 Bob 帮忙挑裤子，虽因已有紫色裤子而犹豫，但在 Bob 建议多色搭配及优先...,莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。
3,test_00004,Emma 因心情不好拒绝 Will 做晚餐，并婉拒其接送提议，决定独自回家。,艾玛很快就会回家，而且她会告诉威尔。
4,test_00005,Ollie 因派对拒绝了晚餐邀请，Jane 随后提议将午餐改为周五课后喝茶，双方达成一致。,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...


In [17]:
# Cell 16: Inspect intermediate outputs and final output

if not df.empty:
    inspection_columns = [
        "id",
        "agent1_semantic_representation_json",
        "agent2_summary_generation_json",
        "agent3_revision_json",
        "final_summary",
    ]

    inspection_df = df[inspection_columns].copy()
    display(inspection_df)
else:
    print("No results found.")

,id,agent1_semantic_representation_json,agent2_summary_generation_json,agent3_revision_json,final_summary
0,test_00001,"{\n ""participants"": [\n ""Hannah"",\n ""Am...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Amanda建议Hannah向Larry索要Betty的号码，尽管Hannah起初不愿，最终...
1,test_00002,"{\n ""participants"": [\n ""Eric"",\n ""Rob""...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Rob 确认脱口秀表演在 YouTube 上，Eric 决定立即观看，Rob 也同意一起看。
2,test_00003,"{\n ""participants"": [\n ""Lenny"",\n ""Bob...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Lenny 请 Bob 帮忙挑裤子，虽因已有紫色裤子而犹豫，但在 Bob 建议多色搭配及优先...
3,test_00004,"{\n ""participants"": [\n ""Will"",\n ""Emma...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Emma 因心情不好拒绝 Will 做晚餐，并婉拒其接送提议，决定独自回家。
4,test_00005,"{\n ""participants"": [\n ""Ollie"",\n ""Jan...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": true,\n ""issues_identi...",Ollie 因派对拒绝了晚餐邀请，Jane 随后提议将午餐改为周五课后喝茶，双方达成一致。
